# 02 - Data Preparation (Preparação de Dados)

**Etapa CRISP-DM**: Data Preparation

**Objetivo**: Validar, limpar e preparar dados educacionais brutos do INEP para as próximas etapas de análise e modelagem.

**Período de Dados**: 2018-2022  
**Unidade Geográfica**: 27 UFs do Brasil

---

## Contexto

Nesta etapa, vamos:

1. **Validar** arquivos de dados brutos do INEP
2. **Tratar** estruturas complexas de planilhas (cabeçalhos aninhados, skiprows)
3. **Processar** múltiplos anos de dados educacionais
4. **Consolidar** em um dataset único e limpo
5. **Validar** consistência e integridade dos dados

Os dados preparados nesta etapa serão integrados com dados socioeconômicos na próxima etapa.

## Importações e Setup

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

---

## Seção 1: Validação de Arquivo INEP (2018)

**Objetivo**: Carregar e validar a estrutura de um arquivo Excel do INEP.

Nesta seção vamos:
- Verificar se o arquivo existe
- Carregar a planilha Excel
- Analisar dimensões (linhas e colunas)
- Identificar colunas importantes (UF, evasão, repetência)
- Visualizar dados brutos

In [2]:
# Configuração de caminhos
# Use pathlib para caminhos relativos ao notebook
from pathlib import Path
data_dir = Path('../data/Raw')
file_path = data_dir / 'TX_REND_BRASIL_REGIOES_UFS_2018.xlsx'
print("="*70)
print("VALIDAÇÃO DE ARQUIVO INEP - TAXA DE RENDIMENTO")
print("="*70)
# Verificar se arquivo existe
if not os.path.exists(file_path):
    print(f"ERRO: Arquivo não encontrado: {file_path}")
    raise ValueError(f"Arquivo não encontrado: {file_path}")
# Ler o Excel com configuração padrão
df = pd.read_excel(file_path)
print(f"\nArquivo carregado: {file_path}")
print(f"Formato: Excel (.xlsx)")
print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")



VALIDAÇÃO DE ARQUIVO INEP - TAXA DE RENDIMENTO

Arquivo carregado: ../data/Raw/TX_REND_BRASIL_REGIOES_UFS_2018.xlsx
Formato: Excel (.xlsx)
Dimensões: 596 linhas x 58 colunas


In [3]:
# Exibir colunas disponíveis
print(f"\nColunas disponíveis ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")


Colunas disponíveis (58):
   1. Unnamed: 0
   2.      Ministério da Educação
   3. Unnamed: 2
   4. Unnamed: 3
   5. Unnamed: 4
   6. Unnamed: 5
   7. Unnamed: 6
   8. Unnamed: 7
   9. Unnamed: 8
  10. Unnamed: 9
  11. Unnamed: 10
  12. Unnamed: 11
  13. Unnamed: 12
  14. Unnamed: 13
  15. Unnamed: 14
  16. Unnamed: 15
  17. Unnamed: 16
  18. Unnamed: 17
  19. Unnamed: 18
  20. Unnamed: 19
  21. Unnamed: 20
  22. Unnamed: 21
  23. Unnamed: 22
  24. Unnamed: 23
  25. Unnamed: 24
  26. Unnamed: 25
  27. Unnamed: 26
  28. Unnamed: 27
  29. Unnamed: 28
  30. Unnamed: 29
  31. Unnamed: 30
  32. Unnamed: 31
  33. Unnamed: 32
  34. Unnamed: 33
  35. Unnamed: 34
  36. Unnamed: 35
  37. Unnamed: 36
  38. Unnamed: 37
  39. Unnamed: 38
  40. Unnamed: 39
  41. Unnamed: 40
  42. Unnamed: 41
  43. Unnamed: 42
  44. Unnamed: 43
  45. Unnamed: 44
  46. Unnamed: 45
  47. Unnamed: 46
  48. Unnamed: 47
  49. Unnamed: 48
  50. Unnamed: 49
  51. Unnamed: 50
  52. Unnamed: 51
  53. Unnamed: 52
  54. Unname

In [4]:
# Exibir primeiras linhas
print("\nPrimeiras 5 linhas:")
print(df.head())


Primeiras 5 linhas:
                                                                                                                                                                                                                     Unnamed: 0  \
0                                                                                                                                                                                                                           NaN   
1                                                                                                                                                                                                                           NaN   
2                                                                                                                                      Taxas de Rendimento Escolar - Brasil, Regiões Geográficas e Unidades da Federação - 2018   
3  Taxas de Rendimento Escolar (Aprovação, Reprovação e Abandono), segu

In [5]:
# Análise das colunas principais
print("\n" + "="*70)
print("ANÁLISE DAS COLUNAS")
print("="*70)

# Identificar colunas por tipo
uf_cols = [c for c in df.columns if any(x in str(c).lower() for x in ['uf', 'estado', 'região', 'geográfica'])]
evasao_cols = [c for c in df.columns if 'evas' in str(c).lower()]
reprovacao_cols = [c for c in df.columns if any(x in str(c).lower() for x in ['reprov', 'repetência'])]
aprovacao_cols = [c for c in df.columns if 'aprova' in str(c).lower()]

print("\nColunas de Localização (UF/Estado/Região):")
if uf_cols:
    for col in uf_cols:
        print(f"  - {col}: {df[col].nunique()} valores únicos")
else:
    print("  NENHUMA coluna de localização encontrada")

print("\nColunas de Evasão:")
if evasao_cols:
    for col in evasao_cols:
        print(f"  - {col}")
else:
    print("  NENHUMA coluna de evasão encontrada")

print("\nColunas de Repetência/Reprovação:")
if reprovacao_cols:
    for col in reprovacao_cols:
        print(f"  - {col}")
else:
    print("  NENHUMA coluna de repetência/reprovação encontrada")

print("\nColunas de Aprovação:")
if aprovacao_cols:
    for col in aprovacao_cols:
        print(f"  - {col}")
else:
    print("  NENHUMA coluna de aprovação encontrada")


ANÁLISE DAS COLUNAS

Colunas de Localização (UF/Estado/Região):
  NENHUMA coluna de localização encontrada

Colunas de Evasão:
  NENHUMA coluna de evasão encontrada

Colunas de Repetência/Reprovação:
  NENHUMA coluna de repetência/reprovação encontrada

Colunas de Aprovação:
  NENHUMA coluna de aprovação encontrada


---

## Seção 2: Tratamento de Estrutura Complexa

**Objetivo**: Lidar com planilhas que têm estruturas complexas (cabeçalhos aninhados, múltiplas linhas de header, etc).

Muitas planilhas do INEP têm informações nas primeiras linhas que não são dados reais. Precisamos:
- Identificar onde começam os dados reais
- Usar skiprows para pular linhas de header inúteis
- Validar que o resultado está correto

In [6]:
print("="*70)
print("TRATAMENTO DE ESTRUTURA COMPLEXA - ARQUIVO INEP")
print("="*70)

# Estratégia 1: Tentar com skiprows diferentes
print("\nTentando diferentes configurações de leitura...\n")

# Tentativa 1: Sem skip (padrão)
try:
    df_padrao = pd.read_excel(file_path)
    print(f"Tentativa 1 (sem skiprows):")
    print(f"  Dimensões: {df_padrao.shape[0]} linhas x {df_padrao.shape[1]} colunas")
    print(f"  Primeira linha: {df_padrao.iloc[0].tolist()[:3]}...")  # Mostrar primeiros 3 valores
except Exception as e:
    print(f"  ERRO: {e}")

# Tentativa 2: Com skiprows=4 (comum em planilhas INEP)
try:
    df_skip4 = pd.read_excel(file_path, skiprows=4)
    print(f"\nTentativa 2 (skiprows=4):")
    print(f"  Dimensões: {df_skip4.shape[0]} linhas x {df_skip4.shape[1]} colunas")
    print(f"  Primeira linha: {df_skip4.iloc[0].tolist()[:3]}...")  # Mostrar primeiros 3 valores
except Exception as e:
    print(f"  ERRO: {e}")

# Escolher a melhor versão
print("\nRecomendação: Usar a leitura que resulte em dados mais estruturados")

TRATAMENTO DE ESTRUTURA COMPLEXA - ARQUIVO INEP

Tentando diferentes configurações de leitura...

Tentativa 1 (sem skiprows):
  Dimensões: 596 linhas x 58 colunas
  Primeira linha: [nan, '      Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira', nan]...

Tentativa 2 (skiprows=4):
  Dimensões: 592 linhas x 58 colunas
  Primeira linha: ['Ano', 'Unidade Geográfica', 'Localização']...

Recomendação: Usar a leitura que resulte em dados mais estruturados


In [7]:
# Usar a versão sem skip se primeira coluna tiver dados, senão usar skip
if str(df.iloc[0, 0]).lower() in ['uf', 'unidade', 'região'] or pd.isna(df.iloc[0, 0]):
    print("Usando skiprows=4 para melhor estrutura...")
    df_clean = pd.read_excel(file_path, skiprows=4)
else:
    print("Estrutura padrão está adequada.")
    df_clean = df.copy()

print(f"\nDataFrame limpo:")
print(f"  Dimensões: {df_clean.shape[0]} linhas x {df_clean.shape[1]} colunas")
print(f"\nColunas após tratamento:")
for i, col in enumerate(df_clean.columns, 1):
    if pd.notna(col) and str(col).strip():
        print(f"  {i:2d}. {col}")

Usando skiprows=4 para melhor estrutura...

DataFrame limpo:
  Dimensões: 592 linhas x 58 colunas

Colunas após tratamento:
   1. Taxas de Rendimento Escolar (Aprovação, Reprovação e Abandono), segundo a Localização e a Dependência Administrativa, nos Níveis de Ensino Fundamental e Médio - Brasil, Regiões Geográficas e Unidades da Federação - 2018 
   2. Unnamed: 1
   3. Unnamed: 2
   4. Unnamed: 3
   5. Unnamed: 4
   6. Unnamed: 5
   7. Unnamed: 6
   8. Unnamed: 7
   9. Unnamed: 8
  10. Unnamed: 9
  11. Unnamed: 10
  12. Unnamed: 11
  13. Unnamed: 12
  14. Unnamed: 13
  15. Unnamed: 14
  16. Unnamed: 15
  17. Unnamed: 16
  18. Unnamed: 17
  19. Unnamed: 18
  20. Unnamed: 19
  21. Unnamed: 20
  22. Unnamed: 21
  23. Unnamed: 22
  24. Unnamed: 23
  25. Unnamed: 24
  26. Unnamed: 25
  27. Unnamed: 26
  28. Unnamed: 27
  29. Unnamed: 28
  30. Unnamed: 29
  31. Unnamed: 30
  32. Unnamed: 31
  33. Unnamed: 32
  34. Unnamed: 33
  35. Unnamed: 34
  36. Unnamed: 35
  37. Unnamed: 36
  38. Unna

In [8]:
# Exibir dados após limpeza
print("\nPrimeiras 10 linhas do DataFrame limpo:")
print(df_clean.head(10))


Primeiras 10 linhas do DataFrame limpo:
  Taxas de Rendimento Escolar (Aprovação, Reprovação e Abandono), segundo a Localização e a Dependência Administrativa, nos Níveis de Ensino Fundamental e Médio - Brasil, Regiões Geográficas e Unidades da Federação - 2018   \
0                                                                                                                                                                                                                          Ano   
1                                                                                                                                                                                                                          NaN   
2                                                                                                                                                                                                                          NaN   
3                                                      

---

## Seção 3: Processamento de Série Temporal (2018-2022)

**Objetivo**: Processar múltiplos arquivos INEP (um por ano) e consolidar em um único DataFrame.

Nesta seção:
- Localizamos todos os arquivos INEP por ano
- Processamos cada arquivo
- Consolidamos os dados
- Validamos integridade e consistência

In [9]:
print("="*70)
print("PROCESSAMENTO DE DADOS INEP - 2018-2022")
print("="*70)

# Configuração
anos = [2018, 2019, 2020, 2021, 2022]
arquivos_encontrados = {}

print("\nVerificando arquivos...")
for ano in anos:
    # Procurar arquivo do ano (padrão: *YYYY*.xlsx ou similar)
    padrao = str(data_dir / f'*{ano}*.xlsx')
    arquivos = glob.glob(padrao)
    
    if arquivos:
        arquivos_encontrados[ano] = arquivos[0]
        print(f"  {ano}: {os.path.basename(arquivos[0])}")
    else:
        print(f"  {ano}: Arquivo não encontrado")

if not arquivos_encontrados:
    print("\nAVISO: Nenhum arquivo encontrado para 2018-2022")
    print("       Baixe os arquivos do INEP e salve em data/Raw/")
else:
    print(f"\nTotal: {len(arquivos_encontrados)} arquivos encontrados")


PROCESSAMENTO DE DADOS INEP - 2018-2022

Verificando arquivos...
  2018: TX_REND_BRASIL_REGIOES_UFS_2018.xlsx
  2019: tx_rend_brasil_regioes_ufs_2019.xlsx
  2020: tx_rend_brasil_regioes_ufs_2020.xlsx
  2021: tx_rend_brasil_regioes_ufs_2021.xlsx
  2022: tx_rend_brasil_regioes_ufs_2022.xlsx

Total: 5 arquivos encontrados


In [10]:
# Função para processar cada arquivo
def processar_arquivo_inep(file_path, ano):
    """
    Processa um arquivo Excel do INEP e retorna DataFrame limpo.
    
    Args:
        file_path: Caminho do arquivo Excel
        ano: Ano do arquivo
    
    Returns:
        DataFrame com colunas padronizadas
    """
    
    try:
        # Tentar com skiprows padrão
        df = pd.read_excel(file_path, skiprows=4)
        
        # Adicionar coluna de ano
        df['Ano'] = ano
        
        # Contar registros (excluindo cabeçalho)
        return df
    
    except Exception as e:
        print(f"    ERRO ao processar {ano}: {e}")
        return None


# Processar todos os arquivos
print("\nProcessando arquivos...\n")
dataframes = {}

for ano, file_path in sorted(arquivos_encontrados.items()):
    df_processado = processar_arquivo_inep(file_path, ano)
    if df_processado is not None:
        dataframes[ano] = df_processado
        print(f"  {ano}: {len(df_processado)} registros, {len(df_processado.columns)} colunas")
    else:
        print(f"  {ano}: Falha no processamento")


Processando arquivos...

  2018: 592 registros, 59 colunas
  2019: 592 registros, 59 colunas
  2020: 592 registros, 59 colunas
  2021: 592 registros, 59 colunas
  2022: 592 registros, 59 colunas


In [11]:
# Consolidar todos os DataFrames
if dataframes:
    print("\nConsolidando dados...")
    df_consolidado = pd.concat(dataframes.values(), ignore_index=True)
    
    print(f"\nDataset consolidado:")
    print(f"  Total de registros: {len(df_consolidado)}")
    print(f"  Total de colunas: {len(df_consolidado.columns)}")
    print(f"  Período: {df_consolidado['Ano'].min()}-{df_consolidado['Ano'].max()}")
    
    print(f"\nDistribuição por ano:")
    print(df_consolidado['Ano'].value_counts().sort_index())
else:
    print("\nAVISO: Nenhum arquivo foi processado com sucesso")


Consolidando dados...

Dataset consolidado:
  Total de registros: 2960
  Total de colunas: 63
  Período: 2018-2022

Distribuição por ano:
Ano
2018    592
2019    592
2020    592
2021    592
2022    592
Name: count, dtype: int64


In [12]:
# Validação de integridade
print("\n" + "="*70)
print("VALIDAÇÃO DE INTEGRIDADE")
print("="*70)

if len(dataframes) > 0:
    # Verificar valores faltantes
    print("\nValores faltantes por coluna (%):\n")
    missing_pct = (df_consolidado.isnull().sum() / len(df_consolidado) * 100).sort_values(ascending=False)
    print(missing_pct[missing_pct > 0])
    
    # Verificar tipos de dados
    print("\nTipos de dados:")
    print(df_consolidado.dtypes)
    
    # Mostrar amostra dos dados
    print("\nAmostra dos dados consolidados:")
    print(df_consolidado.head(10))


VALIDAÇÃO DE INTEGRIDADE

Valores faltantes por coluna (%):

Taxas de Rendimento Escolar (Aprovação, Reprovação e Abandono), segundo a Localização e a Dependência Administrativa, nos Níveis de Ensino Fundamental e Médio - Brasil, Regiões Geográficas e Unidades da Federação - 2018     80.101351
Taxas de Rendimento Escolar (Aprovação, Reprovação e Abandono), segundo a Localização e a Dependência Administrativa, nos Níveis de Ensino Fundamental e Médio - Brasil, Regiões Geográficas e Unidades da Federação - 2021     80.101351
Taxas de Rendimento Escolar (Aprovação, Reprovação e Abandono), segundo a Localização e a Dependência Administrativa, nos Níveis de Ensino Fundamental e Médio - Brasil, Regiões Geográficas e Unidades da Federação - 2022     80.101351
Taxas de Rendimento Escolar (Aprovação, Reprovação e Abandono), segundo a Localização e a Dependência Administrativa, nos Níveis de Ensino Fundamental e Médio - Brasil, Regiões Geográficas e Unidades da Federação - 2019     80.101351
Ta

---

## Conclusões

Nesta etapa de **Data Preparation** realizamos:

1. **Validação** de arquivos INEP (dimensões, colunas, tipos de dados)
2. **Tratamento** de estruturas complexas de planilhas Excel
3. **Processamento** de série temporal 2018-2022
4. **Consolidação** em um dataset único
5. **Validação** de integridade (valores faltantes, consistência)

### Próximas Etapas

Os dados preparados nesta etapa serão:
- Integrados com dados socioeconômicos (IDH, desemprego, PIB) na etapa de **Data Integration**
- Utilizados para treinamento de modelos na etapa de **Modeling**

### Limitações e Considerações

- Qualidade dos dados depende de fontes do INEP
- Possíveis valores faltantes em alguns anos/estados
- Estrutura de colunas pode variar entre anos
- Validação manual pode ser necessária para casos específicos